In [1]:
import openai
from openai import OpenAI
import tiktoken
import logging
import pandas as pd
import os
import time
from tqdm import tqdm
from dotenv import load_dotenv
from pathlib import Path

In [2]:
DEFAULT_SYSTEM_MESSAGE = """You are a scientific visualization expert and prompt engineer.

Create a prompt for an AI image generator that will produce an accurate and visually compelling cover image for a scientific paper.

CRITICAL RULES:
- Promts size should be 50-150 tokens
- Scientific accuracy: Visual elements must correctly represent the scientific concept
- No fictional or decorative elements that contradict the research
- Use correct scientific terminology in the prompt
- Output ONLY the prompt, no extra text
- Image must be in A4 format (portrait orientation, 1:sqrt(2) aspect ratio). Add this information to prompt.

PROMPT COMPONENTS:
1. Core scientific concept (what is being shown)
2. Specific visual details (colors, materials, lighting)
3. Style appropriate for the journal/conference
4. Composition and perspective

EXAMPLES:

Article: "Graphene-Based Battery with 10x Capacity"
Prompt: "Cross-section of graphene layered anode material, lithium ions intercalating between graphene sheets, electron flow visualized as golden energy streams, blue and gray color palette, scientific illustration style, cutaway view, detailed material texture"

Article: "Neural Network Explains Visual Cortex Activity"
Prompt: "Artificial neural network architecture overlaid on primate visual cortex diagram, activation patterns shown as colored heatmaps, connections between nodes mimicking biological pathways, academic illustration style, split-view showing both AI and biology, cool blue to warm red gradient
"""

In [3]:
encoding = tiktoken.get_encoding("cl100k_base")
tokens = encoding.encode(DEFAULT_SYSTEM_MESSAGE)
len(tokens)

294

In [3]:
class Teacher:
    def __init__(self, api_key, log_path, model='deepseek-chat', temperature=0.7, max_tokens=150, n=3, in_cost=0.28, out_cost=0.42, system_message=DEFAULT_SYSTEM_MESSAGE, debug=False):
        self.client = OpenAI(api_key=api_key, base_url='https://api.deepseek.com')
        self.model = model
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.n = n

        self.result = pd.DataFrame(columns=['id'] + [f'prompt#{i}' for i in range(self.n)])
        self.processed = 0

        self.in_tokens = 0
        self.out_tokens = 0
        self.total_cost = 0
        self.in_cost = in_cost
        self.out_cost = out_cost

        self.tiktoken_encoder = tiktoken.get_encoding('cl100k_base')

        self.system_message = {
            "role" : "system",
            "content" : system_message
        }

        dir = os.path.dirname(log_path)
        os.makedirs(dir, exist_ok=True)
        self.logger = logging.getLogger(f"{__name__}.Teacher")
        self.logger.setLevel(logging.DEBUG if debug else logging.INFO)
        if self.logger.handlers:
            self.logger.handlers.clear()
        formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
        file_handler = logging.FileHandler(log_path, encoding='utf-8')
        file_handler.setLevel(logging.DEBUG if debug else logging.INFO)
        file_handler.setFormatter(formatter)
        self.logger.addHandler(file_handler)
        self.logger.propagate = False

        self.logger.info("======================= New model =======================")
        self.logger.debug("Debug mode is ON")
        self.logger.info(f"Teacher initialized with model=\'{model}\', temp={temperature}, n={n}")

    def _call_api(self, messages, max_retries=5):
        base_delay = 15
        for attempt in range(max_retries):
            try:
                all_responses = []
                for n in range(self.n):
                    response = self.client.chat.completions.create(
                        model = self.model,
                        messages=messages,
                        temperature=self.temperature,
                        max_tokens=self.max_tokens,
                    )
                    all_responses.append(response)
                self.logger.debug(f'Response received, choices={len(all_responses)}')
                return all_responses
            except openai.RateLimitError as e:
                wait = base_delay * (2 ** attempt)
                self.logger.warning(f"Rate Limit reached. Waiting {wait} sec. Attempt: {attempt + 1}/{max_retries}")
                time.sleep(wait)
            except Exception as e:
                self.logger.error(f"An error occured during calling API: {e}")
                raise e

    def _prepare_messages(self, title, abstract, id):
       user_content = f"Generate a prompt for:\nTitle: {title}\n\nAbstract: {abstract}"
       messages = [self.system_message, {"role" : "user", "content" : user_content}]
       self.logger.debug(f"Prepared messages for {id}. System length: {len(self.system_message['content'])}, user length: {len(user_content)}")
       return messages

    def _update_stats(self, responses):
        try:
            in_t = responses[0].usage.prompt_tokens
            out_t = 0
            for response in responses:
                out_t += response.usage.completion_tokens
        except Exception as e:
            self.logger.error(f"An error occured during stats updating: {e}")
            raise

        self.in_tokens += in_t
        self.out_tokens += out_t

        cost = (in_t * self.in_cost + out_t * self.out_cost) / 1_000_000
        self.total_cost += cost

        self.logger.debug(f"Stats. Curr: {in_t} + {out_t} = {in_t + out_t} ({cost}), total_in_tokens: {self.in_tokens}, total_out_tokens: {self.out_tokens}, total_cost: {self.total_cost}")

    def generate(self, article):
        self.logger.debug(f"Article: id={article['id']}, title_len={len(article['title'])}, abstract_len={len(article['abstract'])}")
        try:
            messages = self._prepare_messages(article['title'], article['abstract'], article['id'])
            responses = self._call_api(messages)
            self._update_stats(responses)
            new_row = pd.DataFrame({'id' : [article['id']]})
            for i, response in enumerate(responses):
                new_row[f'prompt#{i}'] = response.choices[0].message.content
            self.result = pd.concat([self.result, new_row], ignore_index=True)
        except Exception as e:
            self.logger.error(f'An error occured during response: {e}')
            raise

    def generate_from_dataset(self, dataset, start_index=0, save_path="../prompts/prompts.tsv", checkpoint_path="../checkpoint/check.tsv", checkpoint_every=100, null_processed=False):
        self.logger.info("======================= New session =======================")
        if null_processed:
            self.processed = 0

        dir = os.path.dirname(checkpoint_path)
        os.makedirs(dir, exist_ok=True)
        self.logger.info(f'Starting processing articles from index {start_index}')
        total = dataset.shape[0]
        for i in tqdm(range(start_index, total)):
            article = dataset.iloc[i]
            if (i + 1) % 1000 == 0:
                self.logger.info(f"Processing {i+1}/{total}: {article['id']}")
            self.logger.debug(f"Processing {i+1}/{total}: {article['id']}")
            try:
                self.generate(article)
                if (i + 1) % checkpoint_every == 0:
                    self._save_checkpoint(checkpoint_path, start_index)
            except Exception as e:
                self.logger.error(f"Failed to process {article['id']}: {e}")
                new_row = pd.DataFrame({'id' : [article['id']]})
                for k in range(self.n):
                    new_row[f'prompt#{k}'] = pd.NA
                self.result = pd.concat([self.result, new_row], ignore_index=True)
                continue
        self.logger.info(f"Saving final checkpoint.")
        self._save_checkpoint(checkpoint_path, start_index)
        self.logger.info(f"Completed. Processed {self.processed} / {dataset.shape[0]} articles")
        rows_with_nan = self.result.isna().any(axis=1).sum()
        if rows_with_nan != 0:
            self.logger.warning(f'Rows with nan: {rows_with_nan}')

        dir = os.path.dirname(save_path)
        os.makedirs(dir, exist_ok=True)
        os.rename(checkpoint_path, save_path)
        self.logger.info(f"Saved dataset to {save_path}")

    def _save_checkpoint(self, path, start_idx):
        size = self.result.shape[0]
        idx = start_idx + self.processed
        self.processed += size        
        try:
            self.result.to_csv(path, sep="\t", mode='w' if idx == 0 else 'a', header=True if idx == 0 else False, index=False, encoding='utf-8-sig')
        except Exception as e:
            self.logger.error(f"An error occured during saving checkpoint: {e}")
            raise
        self.result = self.result[0:0]
        self.logger.info(f"Saved checkpoint for {size} articles ({idx}:{idx + size}). Total cost: {self.total_cost}")


In [4]:
BASE_DIR = Path(os.getcwd()).parent
load_dotenv(BASE_DIR / '.env')
API_KEY = os.getenv('DEEPSEEK_API_KEY')

teacher = Teacher(
    api_key=API_KEY,
    log_path='../logs/Teacher.log',
    model='deepseek-chat',
    temperature=0.8,
    max_tokens=150,
    n=2,
    system_message=DEFAULT_SYSTEM_MESSAGE,
    debug=False
)

In [6]:
train1 = pd.read_csv(f"../dataset/train1.tsv", sep="\t")
train1 = train1[0:1]
train1

,Unnamed: 0,id,origin,title,abstract
0,4641,1712.07512v1,41Kmeta_en,Ethical Questions in NLP Research: The (Mis)-U...,Ideas from forensic linguistics are now being ...


In [7]:
teacher.generate_from_dataset(
    dataset=train1,
    start_index=0,
    save_path=f'../prompts/train1.tsv',
    checkpoint_path=f'../checkpoint/train1.tsv',
    checkpoint_every=10
)

100%|██████████| 1/1 [00:14<00:00, 14.90s/it]


In [8]:
val1 = pd.read_csv(f"../dataset/val1.tsv", sep="\t")
val1 = val1[0:1]
val1

,Unnamed: 0,id,origin,title,abstract
0,625,0811.4413v6,41Kmeta_en,A Spectral Algorithm for Learning Hidden Marko...,Hidden Markov Models (HMMs) are one of the mos...


In [11]:
teacher.generate_from_dataset(
    dataset=val1,
    start_index=0,
    save_path=f'../prompts/val1.tsv',
    checkpoint_path=f'../checkpoint/val1.tsv',
    checkpoint_every=10,
    null_processed=True
)

100%|██████████| 1/1 [00:17<00:00, 17.99s/it]
